# EnergyAware AI - Week 2: Train the Router and Evaluate

**Input:** `results.jsonl` and `queries.jsonl` from Week 1 (same Drive folder).
**Output:** a router that looks only at the query text and picks a model tier, plus the quality-vs-energy curve versus baselines.

**Router idea:** for each tier, predict `P(tier answers correctly)` from the query. Then use a threshold `tau`:
send the query to the **cheapest tier whose predicted probability is at least `tau`** (largest tier if none qualifies).
Sweeping `tau` from 0 to 1 gives the user-adjustable quality/energy knob.

**Two evaluation protocols**
- *In-domain:* 5-fold cross-validation over all queries.
- *Out-of-domain:* train on two datasets, test on the held-out third (a stress test).

No GPU is needed for this notebook (CPU is fine).

In [ ]:
!pip -q install sentence-transformers scikit-learn pandas matplotlib

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", None)

SEED = 42
ORDER = ["small", "medium", "large"]          # cheapest -> most expensive

OUT_DIR = globals().get("OUT_DIR")
if OUT_DIR is None:
    if os.path.exists("/kaggle/working"):
        OUT_DIR = "/kaggle/working/energyaware"
    else:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            OUT_DIR = "/content/drive/MyDrive/energyaware"
        except Exception:
            OUT_DIR = "./energyaware"
RESULTS_PATH = f"{OUT_DIR}/results.jsonl"
QUERIES_PATH = f"{OUT_DIR}/queries.jsonl"
print("Reading from:", OUT_DIR)

## 1. Load data and build the per-query table

In [ ]:
res = pd.read_json(RESULTS_PATH, lines=True)
qs = pd.read_json(QUERIES_PATH, lines=True).set_index("qid")

ok    = res.pivot(index="qid", columns="tier", values="correct")[ORDER].astype(float)
E_raw = res.pivot(index="qid", columns="tier", values="energy_j")[ORDER]
E_net = res.pivot(index="qid", columns="tier", values="energy_net_j")[ORDER]

complete = ok.dropna().index.intersection(E_raw.dropna().index).intersection(E_net.dropna().index)
ok, E_raw, E_net = ok.loc[complete], E_raw.loc[complete], E_net.loc[complete]
data = qs.loc[complete].copy()
print(len(data), "queries with all 3 tiers")

ok_a, Er_a, En_a = ok.values, E_raw.values, E_net.values
n = len(data)
ds = data["dataset"].values

print("\nAccuracy:", {k: round(float(v), 3) for k, v in zip(ORDER, ok_a.mean(0))})
print("Mean raw energy (J):", {k: round(float(v), 1) for k, v in zip(ORDER, Er_a.mean(0))})
print("Mean net energy (J):", {k: round(float(v), 1) for k, v in zip(ORDER, En_a.mean(0))})

## 2. Features
Sentence embeddings of the query (MiniLM) plus a few simple surface features. The dataset name is **not** used as a feature.
If the embedding model cannot be downloaded, the notebook falls back to TF-IDF so it still runs (it prints which one was used).

In [ ]:
import re
texts = data["prompt"].tolist()

def embed(texts):
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        X = m.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        return X, "MiniLM sentence embeddings"
    except Exception as e:
        print("Embedding model unavailable, falling back to TF-IDF + SVD:", type(e).__name__)
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        T = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2).fit_transform(texts)
        k = min(128, T.shape[1] - 1)
        return TruncatedSVD(k, random_state=SEED).fit_transform(T), "TF-IDF + SVD"

EMB, emb_name = embed(texts)
print("Embedding:", emb_name, EMB.shape)

def hand_features(t):
    words = t.split()
    digits = sum(c.isdigit() for c in t)
    return [np.log1p(len(t)), np.log1p(len(words)), digits / max(len(t), 1),
            len(re.findall(r"\d+", t)), float("\nA." in t)]

HAND = np.array([hand_features(t) for t in texts])
X = np.hstack([EMB, HAND])
LENGTH = HAND[:, 0]            # used by the length-rule baseline
print("Feature matrix:", X.shape)

## 3. Out-of-fold predictions of `P(correct)` for each tier
Every query is predicted by a model that never saw it (or its dataset, in the out-of-domain protocol).

In [ ]:
def make_model():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=3000))

def fit_predict(train_idx, test_idx):
    P = np.zeros((len(test_idx), len(ORDER)))
    for t in range(len(ORDER)):
        y = ok_a[train_idx, t]
        if y.min() == y.max():                      # constant label: no model needed
            P[:, t] = y[0]
        else:
            P[:, t] = make_model().fit(X[train_idx], y).predict_proba(X[test_idx])[:, 1]
    return P

# In-domain: 5-fold CV
P_in = np.zeros((n, len(ORDER)))
for tr, te in KFold(5, shuffle=True, random_state=SEED).split(X):
    P_in[te] = fit_predict(tr, te)

# Out-of-domain: leave one dataset out
P_ood = np.zeros((n, len(ORDER)))
for d in np.unique(ds):
    te = np.where(ds == d)[0]; tr = np.where(ds != d)[0]
    P_ood[te] = fit_predict(tr, te)

from sklearn.metrics import roc_auc_score
print("AUC of predicting 'tier answers correctly' (higher = router has real signal)\n")
rows = []
for name, P in [("in-domain", P_in), ("out-of-domain", P_ood)]:
    for t, tn in enumerate(ORDER):
        y = ok_a[:, t]
        rows.append({"protocol": name, "tier": tn, "AUC": round(roc_auc_score(y, P[:, t]), 3)})
print(pd.DataFrame(rows).pivot(index="tier", columns="protocol", values="AUC").loc[ORDER])

## 4. Policies and baselines
- **Router:** cheapest tier with `P(correct) >= tau` (largest tier otherwise), sweeping `tau`.
- **Length rule:** shortest queries go to small, the rest to large, sweeping the cut-off.
- **Random mix:** a random fraction goes to small, the rest to large (a straight line between always-small and always-large). A useful router must beat this line.
- **Oracle:** sees the answers, so it is an upper bound only.

In [ ]:
idx = np.arange(n)

def evaluate(choice):
    return (ok_a[idx, choice].mean(), Er_a[idx, choice].mean(), En_a[idx, choice].mean())

def route(P, tau):
    ch = np.full(n, len(ORDER) - 1)
    for t in range(len(ORDER) - 2, -1, -1):          # overwrite so the cheapest qualifying tier wins
        ch[P[:, t] >= tau] = t
    return ch

TAUS = np.concatenate([np.linspace(0, 1, 201), [1.01]])
def router_curve(P):
    return np.array([evaluate(route(P, tau)) for tau in TAUS])

def length_curve():
    rank = LENGTH.argsort().argsort() / n
    return np.array([evaluate(np.where(rank < f, 0, 2)) for f in np.linspace(0, 1, 101)])

def random_curve():
    a = np.array(evaluate(np.zeros(n, int))); b = np.array(evaluate(np.full(n, 2)))
    return np.array([(1 - w) * b + w * a for w in np.linspace(0, 1, 101)])

def oracle_choice():
    ch = np.zeros(n, int)
    for i in range(n):
        for t in range(len(ORDER)):
            if ok_a[i, t]:
                ch[i] = t; break
        else:
            ch[i] = 0                                  # nobody is right: cheapest tier
    return ch

curves = {
    "router (in-domain)": router_curve(P_in),
    "router (out-of-domain)": router_curve(P_ood),
    "length rule": length_curve(),
    "random mix": random_curve(),
}
always = {tn: evaluate(np.full(n, t)) for t, tn in enumerate(ORDER)}
oracle = evaluate(oracle_choice())
print("always-tier (acc, raw J, net J):")
for k, v in always.items(): print(f"  {k:7s} {v[0]:.3f}  {v[1]:.1f}  {v[2]:.1f}")
print(f"  oracle  {oracle[0]:.3f}  {oracle[1]:.1f}  {oracle[2]:.1f}")

## 5. Headline result: energy saved at a given accuracy

In [ ]:
acc_large, Er_large, En_large = always["large"]

def saving(curve, retain, col):
    """Best energy saving (vs always-large) among points keeping >= retain * large accuracy."""
    target = retain * acc_large
    pts = curve[curve[:, 0] >= target]
    if len(pts) == 0:
        return np.nan
    base = Er_large if col == 1 else En_large
    return 100 * (1 - pts[:, col].min() / base)

rows = []
for retain in [1.00, 0.98, 0.95, 0.90]:
    row = {"accuracy kept vs always-large": f"{int(retain*100)}%"}
    for name, c in curves.items():
        row[name + " | raw"] = round(saving(c, retain, 1), 1)
    rows.append(row)
tab = pd.DataFrame(rows).set_index("accuracy kept vs always-large")
print("Energy saved (%) vs always-large, using RAW GPU energy:")
print(tab.filter(like="| raw").rename(columns=lambda c: c.replace(" | raw", "")))

rows = []
for retain in [1.00, 0.98, 0.95, 0.90]:
    row = {"accuracy kept vs always-large": f"{int(retain*100)}%"}
    for name, c in curves.items():
        row[name + " | net"] = round(saving(c, retain, 2), 1)
    rows.append(row)
tab2 = pd.DataFrame(rows).set_index("accuracy kept vs always-large")
print("\nEnergy saved (%) vs always-large, using NET (above-idle) energy:")
print(tab2.filter(like="| net").rename(columns=lambda c: c.replace(" | net", "")))

# Which tiers does the in-domain router use at the ~95% operating point?
target = 0.95 * acc_large
best = None
for tau in TAUS:
    a, er, en = evaluate(route(P_in, tau))
    if a >= target and (best is None or er < best[1]):
        best = (tau, er)
if best:
    ch = route(P_in, best[0])
    print(f"\nIn-domain router at tau={best[0]:.2f} (keeps >=95% of large accuracy): tier usage")
    print(pd.Series(np.array(ORDER)[ch]).value_counts(normalize=True).round(3).to_dict())

## 6. Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, key in zip(axes, ["router (in-domain)", "router (out-of-domain)"]):
    for name, style in [(key, "-"), ("length rule", "--"), ("random mix", ":")]:
        c = curves[name]
        ax.plot(c[:, 1], c[:, 0], style, label=name, lw=2)
    for tn, (a, er, en) in always.items():
        ax.scatter(er, a, s=70, zorder=5); ax.annotate("always " + tn, (er, a), textcoords="offset points", xytext=(6, -12))
    ax.scatter(oracle[1], oracle[0], marker="*", s=220, color="gold", edgecolor="k", zorder=6, label="oracle (upper bound)")
    ax.set_xlabel("Mean GPU energy per query (J, raw)")
    ax.set_title(key); ax.grid(alpha=0.3); ax.legend(loc="lower right")
axes[0].set_ylabel("Accuracy")
plt.tight_layout()
fig_path = f"{OUT_DIR}/pareto_curve.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print("Saved:", fig_path)

## 7. Per-dataset breakdown (in-domain router)
Shows whether the router is just learning "math goes to large" or something finer.

In [ ]:
def per_dataset(P, tau):
    ch = route(P, tau)
    out = []
    for d in np.unique(ds):
        m = ds == d
        out.append({"dataset": d,
                    "accuracy": ok_a[idx[m], ch[m]].mean(),
                    "raw_J": Er_a[idx[m], ch[m]].mean(),
                    "always_large_acc": ok_a[m, 2].mean(),
                    "always_large_J": Er_a[m, 2].mean(),
                    "% to small": 100 * (ch[m] == 0).mean(),
                    "% to medium": 100 * (ch[m] == 1).mean(),
                    "% to large": 100 * (ch[m] == 2).mean()})
    return pd.DataFrame(out).set_index("dataset").round(3)

tau_show = best[0] if best else 0.5
print(f"tau = {tau_show:.2f}")
print(per_dataset(P_in, tau_show))

## Notes for your report
- **Honest framing:** the oracle is an upper bound; the router's real savings are lower. Report the *router* numbers from section 5, and show the oracle only as a ceiling.
- **Raw vs net energy:** raw includes idle GPU power (conservative); net subtracts it. Show both.
- **Out-of-domain** is a stress test: holding out GSM8K removes all math from training, so a drop there is expected and worth discussing.
- **Findings worth stating:** the 3B fp16 tier uses nearly as much energy as the 7B 4-bit tier on a T4, so most of the routing gain comes from choosing small vs (medium/large).
- **Next:** try LightGBM or a fine-tuned small encoder as the router, add more datasets (e.g. HellaSwag, TruthfulQA, a coding set) for a stronger out-of-domain test, and build the Streamlit demo with the `tau` slider.